# Metodo di Newton per l'ottimizzazione: minimi, selle e smorzamento

Questo notebook applica il metodo di Newton alla ricerca di un minimo di una funzione scalare di più variabili, mostrando sia il caso in cui converge senza problemi sia i modi in cui può fallire. Il percorso è organizzato in quattro step:

1. **Step 1 — La funzione da minimizzare, gradiente e Hessiana.** Una quartica non convessa con tre punti stazionari, gradiente e Hessiana calcolati simbolicamente e validati per differenze finite.
2. **Step 2 — Newton smorzato: implementazione e caso regolare.** L'algoritmo, verificato su un punto di partenza dove la Hessiana è definita positiva: convergenza a un minimo genuino.
3. **Step 3 — Punto di partenza patologico: Hessiana indefinita.** Cosa succede partendo esattamente da una sella.
4. **Step 4 — Effetto del passo di smorzamento $\alpha$.** Da un punto vicino alla sella, come $\alpha$ influenza velocità (e, in questo esempio, non la correttezza) della convergenza.

## Perché Newton per l'ottimizzazione

L'ottimizzazione non vincolata — trovare il punto che minimizza una funzione scalare — è alla base di una quantità enorme di problemi applicativi: fitting di modelli ai dati, controllo ottimo, allenamento di modelli di machine learning, progettazione ingegneristica. Il metodo del gradiente (gradient descent) è l'approccio più semplice, ma converge solo linearmente e richiede di scegliere con cura il passo. Applicando l'idea di Newton — linearizzare non $f$ ma il suo gradiente $\nabla f$, usando la Hessiana come "derivata" del gradiente — si ottiene un metodo del second'ordine, con convergenza quadratica vicino a un minimo regolare: molto più rapido, ma anche più esigente (richiede la Hessiana, non solo il gradiente) e più fragile, perché nulla nell'equazione $\nabla f=0$ distingue un minimo da una sella o da un massimo. È proprio questo compromesso — velocità contro robustezza — il filo conduttore di questo notebook.

## Step 1 — La funzione da minimizzare, gradiente e Hessiana

### Gradiente e Hessiana per l'ottimizzazione

Per una funzione scalare $f:\mathbb{R}^n\to\mathbb{R}$ sufficientemente regolare, lo sviluppo di Taylor al second'ordine attorno a un punto $x_k$ è

$$f(x) \approx f(x_k) + \nabla f(x_k)^\top(x-x_k) + \tfrac12(x-x_k)^\top Hf(x_k)(x-x_k),$$

dove $\nabla f(x_k)=\big(\partial f/\partial x_1,\dots,\partial f/\partial x_n\big)^\top$ è il **gradiente** (vettore delle derivate parziali prime) e $Hf(x_k)$ è la **matrice Hessiana** (matrice delle derivate parziali seconde, $\big[Hf\big]_{ij}=\partial^2f/\partial x_i\partial x_j$), simmetrica quando $f$ è di classe $C^2$.

Il gradiente individua i **punti stazionari** di $f$ (quelli in cui $\nabla f=0$, condizione necessaria del prim'ordine per un estremo locale), mentre la Hessiana ne determina la natura tramite il termine quadratico dello sviluppo: negli step successivi useremo esattamente questi due oggetti per costruire l'iterazione di Newton applicata alla ricerca di un minimo (Newton "smorzato").

### Import e librerie

Importiamo `numpy`, `matplotlib.pyplot` e `sympy`. Si riutilizzano l'helper `to_numeric` e l'algoritmo `newton_damped_vett`, creati in `src/symbolic_utils.py` e `src/newton.py`.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from src.symbolic_utils import to_numeric
from src.newton import newton_damped_vett

### La funzione da minimizzare

Si usa la quartica non convessa

$$f(x,y) = x^4+y^4-4xy+1,$$

diversa dalla funzione di lezione ma con la stessa struttura didattica: più punti stazionari, non tutti minimi. Si possono trovare a mano risolvendo $\nabla f=0$:

$$\frac{\partial f}{\partial x}=4x^3-4y=0, \qquad \frac{\partial f}{\partial y}=4y^3-4x=0 \;\Longrightarrow\; y=x^3,\; x=y^3.$$

Sostituendo la prima nella seconda si ottiene $x=(x^3)^3=x^9$, cioè $x^9-x=0 \Rightarrow x(x^8-1)=0$. Le uniche radici reali sono $x=0$ e $x=\pm1$ (le altre sei radici ottave sono complesse). Si ottengono così **tre punti stazionari reali**:

$$(0,0), \qquad (1,1), \qquad (-1,-1),$$

la cui natura (punto di sella o minimo) verrà determinata negli step successivi tramite la Hessiana.

In [ ]:
x_sym = sp.symbols('x0:2')
X = sp.Matrix(x_sym)
fsym = X[0]**4 + X[1]**4 - 4*X[0]*X[1] + 1

grad_sym = fsym.diff(X)
hess_sym = sp.hessian(fsym, X)
print('f(x,y)   =', fsym)
print('grad f   =', grad_sym.T)
print('Hess f   =', hess_sym)

f_num = to_numeric(fsym, [x_sym])
grad_num = to_numeric(grad_sym, [x_sym])
hess_num = to_numeric(hess_sym, [x_sym])

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

xx3d = np.linspace(-1.8, 1.8, 100)
yy3d = np.linspace(-1.8, 1.8, 100)
XX3d, YY3d = np.meshgrid(xx3d, yy3d)
ZZ3d = XX3d**4 + YY3d**4 - 4*XX3d*YY3d + 1

xs_stat = np.array([0, 1, -1])
ys_stat = np.array([0, 1, -1])
zs_stat = xs_stat**4 + ys_stat**4 - 4*xs_stat*ys_stat + 1

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(XX3d, YY3d, ZZ3d, cmap='viridis', alpha=0.85, linewidth=0)
ax.scatter(xs_stat, ys_stat, zs_stat, color='red', s=50, label='punti stazionari')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x,y)')
ax.set_title('Superficie di $f(x,y)=x^4+y^4-4xy+1$')
ax.legend()
plt.show()

### Controllo di correttezza: differenze finite

Prima di usare `grad_num` nell'algoritmo, lo si verifica confrontandolo con una **derivata numerica per differenze finite centrali** di `f_num`,

$$\frac{\partial f}{\partial x_i}(x) \approx \frac{f(x+h e_i)-f(x-h e_i)}{2h},$$

calcolata in un punto generico (non uno dei tre stazionari trovati sopra, per non anticiparne la classificazione).

In [ ]:
pt = np.array([0.7, -0.3])
h = 1e-6

fd_grad = np.array([
    (f_num(pt + h*np.array([1, 0])) - f_num(pt - h*np.array([1, 0]))) / (2*h),
    (f_num(pt + h*np.array([0, 1])) - f_num(pt - h*np.array([0, 1]))) / (2*h),
])
analytic_grad = np.array(grad_num(pt)).flatten()
print('gradiente analitico          =', analytic_grad)
print('gradiente per diff. finite   =', fd_grad)
print('differenza (norma)           =', np.linalg.norm(analytic_grad - fd_grad))

fd_hess = np.zeros((2, 2))
for j in range(2):
    ej = np.zeros(2); ej[j] = 1
    fd_hess[:, j] = (np.array(grad_num(pt + h*ej)).flatten()
                      - np.array(grad_num(pt - h*ej)).flatten()) / (2*h)
analytic_hess = np.array(hess_num(pt))
print('Hessiana analitica            =\n', analytic_hess)
print('Hessiana per diff. finite      =\n', fd_hess)
print('differenza (norma)            =', np.linalg.norm(analytic_hess - fd_hess))

### Le curve di livello di $f$

Un grafico delle curve di livello aiuta a farsi un'idea visiva di dove si trovano i tre punti stazionari individuati sopra, prima di procedere con l'algoritmo.

In [ ]:
xx = np.linspace(-1.8, 1.8, 400)
yy = np.linspace(-1.8, 1.8, 400)
XX, YY = np.meshgrid(xx, yy)
ZZ = XX**4 + YY**4 - 4*XX*YY + 1

plt.figure(figsize=(6, 5))
cs = plt.contour(XX, YY, ZZ, levels=30)
plt.colorbar(cs, label='f(x,y)')
plt.plot([0, 1, -1], [0, 1, -1], 'ro', label='punti stazionari')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Curve di livello di $f(x,y)=x^4+y^4-4xy+1$')
plt.legend()
plt.show()

## Step 2 — Newton smorzato: implementazione e caso regolare

### Dal modello quadratico al passo di Newton

Cercare un minimo di $f$ significa cercare un punto stazionario, cioè risolvere l'equazione vettoriale $\nabla f(x)=0$. Riprendendo lo sviluppo di Taylor al second'ordine visto allo Step 1,

$$f(x) \approx f(x_k) + \nabla f(x_k)^\top(x-x_k) + \tfrac12(x-x_k)^\top Hf(x_k)(x-x_k),$$

il gradiente di questo modello quadratico rispetto a $x$ è $\nabla f(x_k) + Hf(x_k)(x-x_k)$: imponendo che si annulli in $x_{k+1}$ si ottiene il passo di Newton

$$Hf(x_k)\,\Delta x = \nabla f(x_k), \qquad x_{k+1} = x_k - \Delta x,$$

risolvendo a ogni iterazione un sistema lineare nell'incognita $\Delta x$ (non un'inversione esplicita di $Hf$).

A differenza del caso puro, qui si introduce un **fattore di smorzamento** $\alpha\in(0,1]$, aggiornando

$$x_{k+1} = x_k - \alpha\,\Delta x$$

invece di $x_{k+1}=x_k-\Delta x$. È un rimedio semplice — non un vero *line search*, che sceglierebbe $\alpha$ punto per punto minimizzando $f$ lungo la direzione $-\Delta x$ — contro il rischio di passi troppo lunghi quando ci si allontana dalla zona in cui l'approssimazione quadratica di $f$ è affidabile. Con $\alpha=1$ si riottiene Newton puro.

### Implementazione

`newton_damped_vett(grad, hess, x0, alpha, atol, rtol, nmax)` risolve $Hf(x_k)\Delta x=\nabla f(x_k)$ e aggiorna $x_{k+1}=x_k-\alpha\Delta x$. Il criterio d'arresto è sull'incremento normwise relativo **effettivamente applicato**, $\|\alpha\Delta x\|$ (non $\|\Delta x\|$ non smorzato) — coerente con la presenza di $\alpha$, dato che è quello il vero spostamento a ogni iterazione.

### Un caso regolare: partenza vicina al minimo $(1,1)$

Si parte da $x_0=(1.5,\,1.5)$ con $\alpha=0.9$. In questa zona la Hessiana

$$Hf(x,y) = \begin{pmatrix}12x^2 & -4 \\ -4 & 12y^2\end{pmatrix}$$

è definita positiva (in $(1.5,1.5)$: $Hf=\begin{pmatrix}27&-4\\-4&27\end{pmatrix}$, autovalori $31$ e $23$, entrambi positivi), quindi ci si aspetta una convergenza pulita verso il minimo $(1,1)$ individuato allo Step 1.

In [ ]:
x0_reg = np.array([1.5, 1.5])
xs_reg, g_reg, H_reg, n_it_reg, all_err_reg = newton_damped_vett(grad_num, hess_num, x0_reg, alpha=0.9, atol=1e-10, rtol=1e-10, nmax=100)

print('soluzione xs =', xs_reg)
print('grad f(xs) =', g_reg)
print('numero iterate =', n_it_reg)

### Verifica a posteriori: è davvero un minimo?

Trovare un punto con $\nabla f\approx0$ non basta per concludere che sia un minimo: è solo la condizione **necessaria** del prim'ordine. La condizione **sufficiente** del second'ordine richiede che la Hessiana in quel punto sia **definita positiva**, cioè che tutti i suoi autovalori siano positivi. Essendo $Hf$ simmetrica, si usa `np.linalg.eigvalsh` per calcolarne gli autovalori nel punto di convergenza.

In [ ]:
H_xs = np.array(hess_num(xs_reg))
eigvals = np.linalg.eigvalsh(H_xs)
print('Hessiana in xs =\n', H_xs)
print('autovalori =', eigvals)

### Interpretazione: un minimo genuino

Partendo da $x_0=(1.5,1.5)$ con $\alpha=0.9$, `newton_damped_vett` converge in **12 iterazioni** a $x_s=(1.0,\,1.0)$ — esattamente il punto stazionario individuato allo Step 1 — con $\nabla f(x_s)\approx(9.4\times10^{-10},\,9.4\times10^{-10})$, nullo entro la tolleranza richiesta.

La Hessiana lì è $Hf(x_s)=\begin{pmatrix}12&-4\\-4&12\end{pmatrix}$, con autovalori $\{8,\,16\}$: **entrambi positivi**, esattamente come previsto dall'analisi svolta prima di lanciare l'algoritmo. La condizione sufficiente del second'ordine è quindi soddisfatta: $x_s=(1,1)$ è un **minimo locale genuino** di $f$, non solo un punto stazionario qualsiasi — coerente con quanto ci si aspettava partendo da una zona in cui la Hessiana è definita positiva lungo tutto il percorso.

## Step 3 — Punto di partenza patologico: Hessiana indefinita

### Newton non distingue minimi da selle

L'iterazione di Newton per l'ottimizzazione, come vista allo Step 2, risolve $\nabla f(x)=0$: qualunque punto stazionario — minimo, massimo o sella — è per costruzione un punto fisso dell'iterazione, perché lì il gradiente si annulla. Nulla nell'algoritmo stesso distingue a priori questi tre casi: la distinzione richiede di guardare la Hessiana, come già fatto allo Step 2 con il controllo degli autovalori.

### La Hessiana nell'origine è indefinita

Nel punto stazionario $(0,0)$ individuato allo Step 1, la Hessiana è

$$Hf(0,0) = \begin{pmatrix}0&-4\\-4&0\end{pmatrix},$$

con autovalori $\lambda^2-16=0 \Rightarrow \lambda=\pm4$: **segno opposto**, quindi indefinita — $(0,0)$ è una sella, non un minimo. Lo si verifica anche numericamente.

In [ ]:
origin = np.array([0.0, 0.0])
H_origin = np.array(hess_num(origin))
eig_origin = np.linalg.eigvalsh(H_origin)
print('grad f(0,0) =', np.array(grad_num(origin)).flatten())
print('Hessiana in (0,0) =\n', H_origin)
print('autovalori =', eig_origin)

### Newton smorzato da $(0,0)$, con tre valori di $\alpha$

Si lancia `newton_damped_vett` esattamente da $x_0=(0,0)$ con $\alpha\in\{0.5,\,0.9,\,1.0\}$, per confrontare cosa succede nei tre casi.

In [ ]:
print(f"{'alpha':<8}{'xs':<20}{'n_it':<6}")
for alpha in [0.5, 0.9, 1.0]:
    xs_a, g_a, H_a, n_it_a, _ = newton_damped_vett(grad_num, hess_num, origin, alpha=alpha, atol=1e-10, rtol=1e-10, nmax=100)
    xs_str = f"({xs_a[0]:.4f}, {xs_a[1]:.4f})"
    print(f"{alpha:<8}{xs_str:<20}{n_it_a:<6}")

### Perché una Hessiana indefinita è pericolosa (e cosa succede qui)

In generale, quando $Hf(x_k)$ non è definita positiva, la direzione di Newton $-\Delta x=-Hf(x_k)^{-1}\nabla f(x_k)$ **non è garantita essere una direzione di discesa** per $f$: la condizione che garantisce $\nabla f(x_k)^\top(-\Delta x)<0$ (cioè che muoversi in quella direzione faccia effettivamente diminuire $f$ al prim'ordine) è proprio che $Hf(x_k)$ sia definita positiva. Se non lo è, il passo di Newton può puntare verso un punto in cui $f$ aumenta, o verso un altro punto stazionario qualsiasi — non necessariamente verso un minimo.

Nel nostro caso specifico, però, il fenomeno si manifesta in un modo ancora più diretto: $(0,0)$ non è solo un punto a Hessiana indefinita, è **già** un punto stazionario ($\nabla f(0,0)=(0,0)$, verificato sopra). Il passo di Newton è $\Delta x = Hf(0,0)^{-1}\cdot 0 = 0$: nullo, indipendentemente da $\alpha$. Lo smorzamento non ha nulla da correggere, perché qui il problema non è "il passo è troppo lungo" (che è ciò che $\alpha$ è pensato per correggere) ma "la direzione stessa è assente". Il risultato, per tutti e tre i valori di $\alpha$ testati, è identico: **convergenza istantanea (1 iterazione) esattamente in $(0,0)$**.

Questo è un falso positivo silenzioso: il criterio d'arresto (incremento nullo) è soddisfatto, ma il punto trovato è una sella, non un minimo — l'algoritmo non ha alcun modo di accorgersene da solo. È esattamente per questo che il controllo degli autovalori della Hessiana introdotto allo Step 2 non è un passaggio opzionale o pro forma: è l'unico modo per scoprire che la "convergenza" ottenuta non corrisponde a un minimo.

### Interpretazione

I numeri confermano l'analisi: $\nabla f(0,0)=(0,0)$ esattamente, $Hf(0,0)$ ha autovalori $\{-4,\,4\}$ (segno opposto, indefinita), e per **tutti e tre** i valori di $\alpha\in\{0.5,0.9,1.0\}$ l'algoritmo restituisce $x_s=(0,0)$ in **1 sola iterazione** — un comportamento chiaramente diverso da quello dello Step 2, ma non nel modo che ci si aspetterebbe (instabilità o mancata convergenza): qui il fallimento è più subdolo, perché il criterio di arresto è soddisfatto ma **il punto non è un minimo**. Senza il controllo degli autovalori, non ci sarebbe alcun segnale — nell'output dell'algoritmo — che qualcosa non va: la lezione di questo step è che la convergenza dell'iterazione, da sola, non certifica mai di aver trovato un minimo.

## Step 4 — Effetto del passo di smorzamento alpha

A differenza dello Step 3, qui si parte da un punto dove il gradiente **non** è nullo, quindi l'algoritmo compie passi genuini: il suo comportamento può davvero dipendere da $\alpha$, non essere identicamente nullo per costruzione. L'obiettivo è quantificare il compromesso velocità/stabilità legato alla scelta di $\alpha$.

### Un punto vicino alla sella, ma diverso

Si sceglie $x_0=(0.1,\,-0.1)$: vicino alla sella $(0,0)$, ma con gradiente non nullo. La Hessiana resta indefinita anche qui, perché i termini $12x^2=0.12$ e $12y^2=0.12$ sono ancora trascurabili rispetto a $\pm4$.

In [ ]:
x0_near = np.array([0.1, -0.1])
H_near = np.array(hess_num(x0_near))
eig_near = np.linalg.eigvalsh(H_near)
print('grad f(x0) =', np.array(grad_num(x0_near)).flatten())
print('Hessiana in x0 =\n', H_near)
print('autovalori =', eig_near)

### Griglia di valori di $\alpha$

Da questo stesso punto, si lancia `newton_damped_vett` per $\alpha=0.1,0.2,\dots,1.0$, registrando il numero di iterazioni (con un `nmax` più alto del solito, per non confondere "lento" con "non convergente") e il punto di arrivo — non è scontato che l'algoritmo si allontani verso un minimo vero: partendo così vicino alla sella, potrebbe anche essere semplicemente riattratto verso la sella stessa.

In [ ]:
nmax_sweep = 200
alphas = np.arange(0.1, 1.0001, 0.1)
n_its = []
xs_list = []

print(f"{'alpha':<8}{'xs':<24}{'n_it':<8}")
for alpha in alphas:
    xs_a, g_a, H_a, n_it_a, _ = newton_damped_vett(grad_num, hess_num, x0_near, alpha=alpha,
                                                     atol=1e-10, rtol=1e-10, nmax=nmax_sweep)
    n_its.append(n_it_a)
    xs_list.append(xs_a)
    converged = 'no' if n_it_a >= nmax_sweep else 'si'
    xs_str = f"({xs_a[0]:.4f}, {xs_a[1]:.4f})"
    print(f"{alpha:<8.1f}{xs_str:<24}{n_it_a:<8}{'' if converged=='si' else '  (non convergente)'}")

### Iterazioni in funzione di $\alpha$

Si visualizza il numero di iterazioni in funzione di $\alpha$ (scala logaritmica sull'asse delle iterazioni, dato l'ampio intervallo di valori).

In [ ]:
plt.figure(figsize=(6, 4))
plt.semilogy(alphas, n_its, 'o-')
plt.xlabel(r'$\alpha$')
plt.ylabel('numero di iterazioni')
plt.title(r'Iterazioni vs $\alpha$, da $x_0=(0.1,-0.1)$')
plt.grid(True, which='both', alpha=0.3)
plt.show()

### Interpretazione: qui $\alpha$ non previene un'instabilità, ne determina solo la velocità

Il risultato è netto ma diverso da quanto ci si potrebbe aspettare a priori: per **tutti** i valori di $\alpha$ testati (da $0.1$ a $1.0$) l'algoritmo converge — nessuna instabilità, nessuna non convergenza entro `nmax`. Ma converge sempre e solo a $(0,0)$, la sella, mai a uno dei due minimi $(\pm1,\pm1)$. Il numero di iterazioni però varia enormemente, da **180** (con $\alpha=0.1$) a sole **4** (con $\alpha=1.0$, Newton puro), diminuendo in modo pressoché monotono al crescere di $\alpha$.

La spiegazione è coerente con quanto visto allo Step 3: vicino all'origine i termini quartici $x^4,y^4$ sono trascurabili rispetto al termine misto $-4xy$, quindi $f$ vi si comporta come una forma quadratica pura con Hessiana $\begin{pmatrix}0&-4\\-4&0\end{pmatrix}$ — e per una funzione esattamente quadratica, Newton converge al suo (unico) punto stazionario, cioè la sella, indipendentemente dal punto di partenza all'interno di questa regione. Uno smorzamento più forte ($\alpha$ piccolo) non evita questa attrazione: la rallenta soltanto, perché ogni passo è più corto e ne servono di più per percorrere la stessa distanza verso $(0,0)$.

In questo esempio, quindi, il "miglior compromesso" tra i valori testati è semplicemente $\alpha=1.0$: converge più rapidamente (4 iterazioni) e, non dovendo prevenire alcuna instabilità (che qui non si manifesta), non c'è motivo di smorzare il passo. Ma è una vittoria di velocità, non di correttezza — il punto trovato resta una sella in tutti i casi, un promemoria che nessuna scelta di $\alpha$ può da sola risolvere il problema strutturale di partire troppo vicino a un punto stazionario che non è un minimo: per questo serve un punto di partenza in una zona diversa (come allo Step 2), non un $\alpha$ diverso.

## Conclusioni e limiti

I quattro step di questo notebook tracciano un percorso coerente, dalla convergenza ideale ai suoi punti deboli.

- **Condizioni di ottimalità del second'ordine.** $\nabla f(x)=0$ è solo la condizione **necessaria** del prim'ordine: la soddisfano indifferentemente minimi, massimi e selle. La condizione **sufficiente** che certifica un minimo è che la Hessiana in quel punto sia **definita positiva** (tutti gli autovalori positivi) — allo Step 2, in $(1,1)$, gli autovalori $\{8,16\}$ hanno confermato un minimo genuino; allo Step 3, in $(0,0)$, gli autovalori $\{-4,4\}$ hanno smascherato una sella che l'algoritmo, da solo, aveva già dichiarato "raggiunta".
- **Perché Newton puro può fallire lontano dall'ottimo.** Risolvendo solo $\nabla f=0$, senza guardare la curvatura, Newton è attratto indifferentemente da qualunque punto stazionario. Lo Step 3 lo ha mostrato nel modo più diretto possibile — un falso positivo istantaneo in $(0,0)$ — e lo Step 4 lo ha confermato da un punto vicino ma non coincidente con la sella: tutti gli $\alpha$ testati convergevano, ma sempre alla sella, mai a un minimo.
- **Ruolo dello smorzamento $\alpha$.** È un rimedio semplice — non un vero *line search*, che cercherebbe punto per punto il passo che minimizza $f$ lungo la direzione di Newton — contro il rischio di passi troppo lunghi quando l'approssimazione quadratica di $f$ non è affidabile. Lo Step 4 ne ha chiarito anche il limite: quando la Hessiana non è definita positiva, nessun $\alpha\in(0,1]$ trasforma una direzione di discesa non garantita in una garantita — $\alpha$ regola la *velocità* con cui si converge, non *dove* si converge.

Un limite comune a tutti gli esperimenti: `newton_damped_vett`, così come implementato qui, non ha alcun meccanismo per rilevare o correggere una curvatura negativa (a differenza di varianti più sofisticate come Newton modificato o i metodi trust-region, che perturbano la Hessiana o limitano esplicitamente il passo quando non è definita positiva), né alcuna garanzia di convergenza globale. La responsabilità di scegliere un punto di partenza in una zona ragionevole — e di verificare sempre, a posteriori, che il punto trovato sia davvero un minimo — resta interamente di chi usa l'algoritmo.